# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [1]:
## (1) Combining & Cleaning Data

import pandas as pd
import numpy as np

# 1. Load datasets
url1 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv" # Version used in the NBs before
url2 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv" # updates and changes via two NEW CSV files 
url3 = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"

df1 = pd.read_csv(url1)
df2 = pd.read_csv(url2)
df3 = pd.read_csv(url3)

# 2. Standardize column names and combine 
def standardize_columns(df):
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    # Rename columns that differ between the 3 versions 
    df = df.rename(columns={'st': 'state'}) 
    return df

df1 = standardize_columns(df1)
df2 = standardize_columns(df2)
df3 = standardize_columns(df3)

# 3. Combine the datasets
combined_df = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

# 4. Clean the merged datasets
def clean_data(df):
    # Drop duplicated rows and reset index
    df = df.drop_duplicates()  
    df = df.reset_index(drop=True) 

    # Clean up 'customer_lifetime_value' to remove % and $ signs
    if 'customer_lifetime_value' in df.columns and df['customer_lifetime_value'].dtype == 'object':
        df['customer_lifetime_value'] = df['customer_lifetime_value'].replace('[\$,%]', '', regex=True)
        # Convert to float, coercing any leftover weird strings to NaN
        df['customer_lifetime_value'] = pd.to_numeric(df['customer_lifetime_value'], errors='coerce')

    # Fill categorical with 'Unknown', numeric with the median
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna('Unknown')
        else:
            df[col] = df[col].fillna(df[col].median())
            
    return df

clean_combined_df = clean_data(combined_df)
print(f"Combined and cleaned data shape: {clean_combined_df.shape}")

Combined and cleaned data shape: (9135, 11)


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [2]:
# (2) Structuring the combined data

url_marketing = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
marketing_df = pd.read_csv(url_marketing) # Load the cleaned csv "Marketing" dataset

# 1. Total revenue for each channel
# 'customer_lifetime_value' just means total sales over time -- CLV 
revenue_by_channel = pd.pivot_table(
    data=marketing_df,
    values='customer_lifetime_value', 
    index='sales_channel',
    aggfunc='sum'
).round(2)
# Rename for clarity - AI 
revenue_by_channel.columns = ['Total Revenue']
print("Total Revenue by Sales Channel")
print(revenue_by_channel)

Total Revenue by Sales Channel
               Total Revenue
sales_channel               
Agent            33057887.85
Branch           24359201.21
Call Center      17364288.37
Web              12697632.90


In [3]:
# 2. Average customer lifetime value per gender and education level
clv_gender_edu = pd.pivot_table(
    data=marketing_df,
    values='customer_lifetime_value',
    index='education',
    columns='gender',
    aggfunc='mean'
).round(2)

print("\nAverage CLV by Gender and Education")
print(clv_gender_edu)


Average CLV by Gender and Education
gender                      F        M
education                             
Bachelor              7874.27  7703.60
College               7748.82  8052.46
Doctor                7328.51  7415.33
High School or Below  8675.22  8149.69
Master                8157.05  8168.83


# NOTES:      
    - As might be expected, the marketing channels "Agent" (33 million total revenue) and "Branch" (24m) dominate, indicating that personal contact is preferred when choosing insurance plans or contracts. Website (12m) and Call center (17m) have lower revenue in total, so these channels are cheaper to operate but typically have much lower conversion rates. 

    - With female customers, ORDINAL Educational level is more closely correlated with CLV (lifetime revenue from customer), with the notable exception of the highest ed level, doctor/PhD. Men with PhD were also an outlier in that they were the lowest, not the highest CLV cluster. And the Mens' data is altogether more varied, so Lifetime Value =revenue is less tied to their educational status than for the female customers. Possible reasons: Smaller, less representative number of Phd level customers in total? Gender workplace and societal differences?  

        HS or below     > College       > Bachelor      > Master    > Doctor
CLV/F    5.highest          2.              3.          4.          1.lowest
CLV/M     4.                3.              2.          5.          1. lowest

1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

# Look up LONG format table -- Wide v. long format just means a pivot form like 

WIDE                                                LONG
index   Var1    Var2    Var3            VS          index   Var     Value    
A       88      12      22                          A       Var1    88   
B                                                   A       Var2    12
C                                                   A       Var3    22
                                                    B 
                                                    B etc. 